In [2]:
import json
import re
from pathlib import Path

import numpy as np
from tensorflow import keras

from action import execute_intent


BASE_PATH = "/home/sudhanshu_bajpai/projects/Voice_assistant/data/"
MODEL_PATH = BASE_PATH + "intent_model.keras"
METADATA_PATH = BASE_PATH + "intent_metadata.json"

CONFIDENCE_THRESHOLD = 0.70


model = keras.models.load_model(MODEL_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

vocab = metadata["vocab"]
intents = metadata["intents"]
MAX_LEN = metadata["max_len"]


def tokenize(text):
    return re.findall(r"[a-zA-Z0-9']+", text.lower())


def encode_sentence(sentence):

    tokens = tokenize(sentence)

    ids = []

    for token in tokens:
        ids.append(
            vocab.get(
                token,
                vocab["<UNK>"]
            )
        )

    ids = ids[:MAX_LEN]

    while len(ids) < MAX_LEN:
        ids.append(vocab["<PAD>"])

    return np.array([ids], dtype=np.int32)


def predict_intent(sentence):

    x = encode_sentence(sentence)

    probabilities = model.predict(
        x,
        verbose=0
    )[0]

    prediction = np.argmax(probabilities)

    confidence = probabilities[prediction]

    intent = intents[prediction]

    return intent, confidence


def process_command(command):

    print(f"\nCommand: {command}")

    intent, confidence = predict_intent(command)

    print(f"Intent: {intent}")
    print(f"Confidence: {confidence:.2%}")

    if confidence < CONFIDENCE_THRESHOLD:

        print(
            "I'm not confident enough to execute that command."
        )

        return True

    print("Executing...")

    return execute_intent(intent, command)


def main():

    print("AI PC Agent started.")

    while True:

        command = input("\nYou: ")

        should_continue = process_command(command)

        if not should_continue:
            print("Assistant stopped.")
            break


if __name__ == "__main__":
    main()

AI PC Agent started.

Command: I ant to make some notes
Intent: MAKE_NOTES
Confidence: 99.88%
Executing...
Opening Notepad...

Command: end
Intent: STOP_ASSISTANT
Confidence: 99.87%
Executing...
Assistant stopped.
